# 🏦 Loan Prepayment Risk Prediction (Kaggle Style)
Binary classification using LightGBM, SHAP, and Gradio UI.

In [ ]:
!pip install lightgbm shap gradio --quiet

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import shap
import gradio as gr


In [ ]:
np.random.seed(222)
n = 1200
df = pd.DataFrame({
    "loan_amount": np.random.randint(5000, 300000, n),
    "interest_rate": np.random.uniform(2.5, 12.0, n),
    "term_months": np.random.choice([120, 180, 240, 360], n),
    "borrower_age": np.random.randint(25, 65, n),
    "credit_score": np.random.randint(580, 850, n),
    "prepaid": np.random.choice([0, 1], n, p=[0.7, 0.3])
})


In [ ]:
X = df.drop(columns="prepaid")
y = df["prepaid"]
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=222)
model = lgb.LGBMClassifier()
model.fit(X_train, y_train)
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))


In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values[1], X_test)


In [ ]:
def predict_prepayment(loan_amount, interest_rate, term_months, borrower_age, credit_score):
    row = pd.DataFrame([[loan_amount, interest_rate, term_months, borrower_age, credit_score]], columns=X.columns)
    prob = model.predict_proba(row)[0][1]
    return f"Prepayment Risk: {prob:.2%}"

gr.Interface(
    fn=predict_prepayment,
    inputs=[
        gr.Number(label="Loan Amount ($)"),
        gr.Slider(2.5, 12.0, label="Interest Rate (%)"),
        gr.Dropdown([120, 180, 240, 360], label="Term (Months)"),
        gr.Slider(18, 80, label="Borrower Age"),
        gr.Slider(580, 850, label="Credit Score")
    ],
    outputs="text",
    title="Loan Prepayment Risk Predictor"
).launch()
